In [ ]:
import os
import sys

sys.path.append(os.path.abspath(os.path.dirname(os.getcwd())))

import re
import json
import platform
import subprocess
import numpy as np
import pandas as pd
import onnxruntime as ort
import matplotlib.pyplot as plt
from enum import Enum
from pathlib import Path
from dataclasses import dataclass, replace
from src.metrics.bjontegaard_metric import calculate_bd_rate
from conversion.types import (
    ModelType,
    TargetDevice,
    RuntimeParams,
    CoremlComputeUnits,
    OpenvinoDevice,
    OnnxExecutionProvider,
)
from conversion import (
    full_model_factory,
    split_full_model,
    exporter_factory,
    load_split_model,
    ModelTester,
)
from conversion.utils import download_job_outputs, parse_metrics

pd.options.display.float_format = "{:,.4g}".format

## Available runtimes

In [ ]:
def apple_cpu_chip():
    if sys.platform != "darwin":
        return None
    brand = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
    m = re.search(r"\bApple\s+(M\d)\b", brand)
    return m.group(1) if m else None

In [ ]:
print(f"Processor: {platform.processor()}, Apple chip: {apple_cpu_chip()}")
print(f"Available execution providers: {ort.get_available_providers()}")

has_qualcomm_x_elite = platform.processor() in [
    "ARMv8 (64-bit) Family 8 Model 1 Revision 201, Qualcomm Technologies Inc"
]
has_intel_lnl = platform.processor() in ["Intel64 Family 6 Model 189 Stepping 1, GenuineIntel"]
has_intel_ptl = platform.processor() in ["Intel64 Family 6 Model 204 Stepping 0, GenuineIntel"]
has_apple_m3 = sys.platform == "darwin" and apple_cpu_chip() in ["M1", "M2", "M3"]
has_apple_m5 = sys.platform == "darwin" and apple_cpu_chip() in ["M5"]

print(f"{has_qualcomm_x_elite=}, {has_intel_lnl=}, {has_intel_ptl=} {has_apple_m3=}, {has_apple_m5=}")

## Params

In [ ]:
class TestType(str, Enum):
    DIVERGENCE = "divergence"
    PROACTIVE_LTR_RECOVERY = "proactive_ltr_recovery"


class Dataset(str, Enum):
    VCD_360P = "vcd_360p"
    VCD_540P = "vcd_540p"


@dataclass
class DatasetConfig:
    test_config_path: str
    anchor_name: str
    anchor_metrics_path: str


DATASET_CONFIGS = {
    Dataset.VCD_360P: DatasetConfig(
        test_config_path="yuv/640x360_30fps/VCD-640x360_30fps.json",
        anchor_name="intel_hw_hevc_lp",
        anchor_metrics_path="benchmark_test/anchor/VCD_640x360_30fps_full/intel_hw_hevc_lp.json",
    ),
    Dataset.VCD_540P: DatasetConfig(
        test_config_path="yuv/960x540_30fps/VCD-960x540_30fps.json",
        anchor_name="intel_hw_hevc_lp",
        anchor_metrics_path="benchmark_test/anchor/VCD_960x540_30fps/intel_hw_hevc_lp.json",
    ),
}


@dataclass
class TestConfig:
    model_version: str
    encoder_name: str
    decoder_name: str
    model_type: ModelType
    target_device: TargetDevice
    runtime_params: RuntimeParams = RuntimeParams()
    encoded_by: str | None = None
    use_decoder: bool = True
    runtime_available: bool = True
    iframe_period: int | None = None
    ltr_start_idx: int | None = None
    ltr_period: int | None = None
    num_clips_limit: int = 1000

    @property
    def test_name(self) -> str:
        return f"{self.model_version}-{self.iframe_period or 'd'}-{self.ltr_period or 'd'}-{self.ltr_start_idx or 'd'}-{self.encoder_name}-{self.decoder_name}-{self.num_clips_limit}s"


model_version = "dmc61sbr_mini_reglu"
model_width = 960
model_height = 544
dataset = Dataset.VCD_540P
dataset_config = DATASET_CONFIGS[dataset]
test_config_path = dataset_config.test_config_path
use_cache = True
metrics_bit_depth = 8
test_type: TestType = TestType.DIVERGENCE

## Compile test configs

In [ ]:
test_configs: dict[str, TestConfig] = {}
if test_type == TestType.DIVERGENCE:
    num_clips_limit = 40
    iframe_period = None
    ltr_start_idx = None
    ltr_period = None

    test_matrix_items = [
        # (
        #     "apple_cpu",
        #     ModelType.COREML,
        #     TargetDevice.APPLE,
        #     RuntimeParams(coreml_compute_units=CoremlComputeUnits.CPU),
        #     has_apple_m3,
        # ),
        (
            "apple_gpu",
            ModelType.COREML,
            TargetDevice.APPLE,
            RuntimeParams(coreml_compute_units=CoremlComputeUnits.GPU),
            has_apple_m3,
        ),
        (
            "apple_npu",
            ModelType.COREML,
            TargetDevice.APPLE,
            RuntimeParams(coreml_compute_units=CoremlComputeUnits.NPU),
            has_apple_m3,
        ),
        # (
        #     "intel_cpu",
        #     ModelType.OPENVINO,
        #     TargetDevice.INTEL,
        #     RuntimeParams(openvino_device=OpenvinoDevice.CPU),
        #     has_intel_lnl,
        # ),
        (
            "intel_gpu",
            ModelType.OPENVINO,
            TargetDevice.INTEL,
            RuntimeParams(openvino_device=OpenvinoDevice.GPU),
            has_intel_lnl,
        ),
        (
            "intel_npu",
            ModelType.OPENVINO,
            TargetDevice.INTEL,
            RuntimeParams(openvino_device=OpenvinoDevice.NPU),
            has_intel_lnl,
        ),
        (
            "qualcomm_npu",
            ModelType.ONNX,
            TargetDevice.QUALCOMM,
            RuntimeParams(onnx_execution_provider=OnnxExecutionProvider.QNN),
            has_qualcomm_x_elite,
        ),
        (
            "apple_gpu_m5",
            ModelType.COREML,
            TargetDevice.APPLE,
            RuntimeParams(coreml_compute_units=CoremlComputeUnits.GPU),
            has_apple_m5,
        ),
        (
            "apple_npu_m5",
            ModelType.COREML,
            TargetDevice.APPLE,
            RuntimeParams(coreml_compute_units=CoremlComputeUnits.NPU),
            has_apple_m5,
        ),
        (
            "intel_gpu_plt",
            ModelType.OPENVINO,
            TargetDevice.INTEL,
            RuntimeParams(openvino_device=OpenvinoDevice.GPU),
            has_intel_ptl,
        ),
        (
            "intel_npu_plt",
            ModelType.OPENVINO,
            TargetDevice.INTEL,
            RuntimeParams(openvino_device=OpenvinoDevice.NPU),
            has_intel_ptl,
        ),
    ]

    encoded_by_list = []
    for name, model_type, target_device, runtime_params, runtime_available in test_matrix_items:
        t = TestConfig(
            model_version,
            name,
            name,
            model_type,
            target_device,
            runtime_params,
            runtime_available=runtime_available,
            iframe_period=iframe_period,
            ltr_start_idx=ltr_start_idx,
            ltr_period=ltr_period,
            num_clips_limit=num_clips_limit,
        )
        test_configs[t.test_name] = t
        encoded_by_list.append(t.test_name)

    for i, (
        encoder_name,
        encoder_model_type,
        encoder_target_device,
        encoder_runtime_params,
        encoder_runtime_available,
    ) in enumerate(test_matrix_items):
        for j, (
            decoder_name,
            decoder_model_type,
            decoder_target_device,
            decoder_runtime_params,
            decoder_runtime_available,
        ) in enumerate(test_matrix_items):
            if i >= j:
                continue
            encoded_by = encoded_by_list[i]
            t = replace(
                test_configs[encoded_by],
                encoder_name=encoder_name,
                decoder_name=decoder_name,
                model_type=decoder_model_type,
                target_device=decoder_target_device,
                encoded_by=encoded_by,
                runtime_params=decoder_runtime_params,
                runtime_available=decoder_runtime_available,
            )
            test_configs[t.test_name] = t

elif test_type == TestType.PROACTIVE_LTR_RECOVERY:
    num_clips_limit = 5
    for iframe_period, ltr_period, ltr_start_idx in [
        (32, None, 0),
        # -----------------
        (64, None, 0),
        (64, 32, 0),
        (64, 32, 8),
        (64, 32, 16),
        # -----------------
        (128, None, 0),
        (128, 64, 0),
        (128, 64, 8),
        (128, 64, 16),
        (128, 32, 0),
        (128, 32, 8),
        (128, 32, 16),
        # -----------------
        (256, None, 0),
        (256, 128, 0),
        (256, 128, 8),
        (256, 128, 16),
        (256, 64, 0),
        (256, 64, 8),
        (256, 64, 16),
        (256, 32, 0),
        (256, 32, 8),
        (256, 32, 16),
    ]:
        baseline_test = TestConfig(
            model_version=model_version,
            model_type=ModelType.COREML,
            target_device=TargetDevice.APPLE,
            encoder_name="coreml_gpu",
            decoder_name="coreml_gpu",
            runtime_params=RuntimeParams(coreml_compute_units=CoremlComputeUnits.GPU),
            iframe_period=iframe_period,
            ltr_period=ltr_period,
            ltr_start_idx=ltr_start_idx,
            num_clips_limit=num_clips_limit,
        )

        divergence_test = replace(
            baseline_test,
            decoder_name="coreml_npu",
            runtime_params=RuntimeParams(coreml_compute_units=CoremlComputeUnits.NPU),
            encoded_by=baseline_test.test_name,
        )

        for t in [baseline_test, divergence_test]:
            test_configs[t.test_name] = t

In [ ]:
def azureml_metrics(job_name: str, exp_name: str = "dcvc-dc") -> Path:
    return download_job_outputs(f"checkpoints/{exp_name}/{job_name}/metrics_VCD-960x540_30fps.json")


comparison_metric_files: dict[str, Path | str] = {
    dataset_config.anchor_name: download_job_outputs(dataset_config.anchor_metrics_path),
    # "dmc61sbr_lrelu, azureml": azureml_metrics("my_job_name"),
}

# Prepare models

In [ ]:
if sys.platform == "darwin":
    supported_model_types = [ModelType.COREML, ModelType.ONNX, ModelType.OPENVINO]
elif platform.system() == "Windows" and platform.machine() == "ARM64":
    supported_model_types = [ModelType.ONNX]
else:
    supported_model_types = [ModelType.ONNX, ModelType.OPENVINO]

model_paths = {}
for model_version, model_type, target_device in set(
    [(t.model_version, t.model_type, t.target_device) for t in test_configs.values()]
):
    if model_type not in supported_model_types:
        print(f"Skipping unsupported model type {model_type} on this platform")
        continue
    full_model = full_model_factory(model_version)
    split_model = split_full_model(full_model, model_width=model_width, model_height=model_height)
    exporter = exporter_factory(
        split_model=split_model,
        model_type=model_type,
        target_device=target_device,
        output_path="./output/divergence_test/models",
        skip_if_exists=use_cache,
        frame_count=1,
    )
    model_paths[model_version, model_type, target_device] = exporter.run()

## Run divergence tests

In [ ]:
results_path = Path("./output/divergence_test/results/").resolve()
test_result_files: dict[str, Path | str] = {}
for test_name, test_config in test_configs.items():
    print(f"Running test: {test_name}")

    # Return cached results if they exist
    test_results_path = results_path / f"{test_name}" / "validation_test_results.json"
    if use_cache and test_results_path.exists():
        print("Test results already exist, using saved results")
        test_result_files[test_name] = test_results_path
        continue

    # Skip tests that require unavailable runtimes
    if not test_config.runtime_available:
        print("Skipping test because model type is not available")
        continue

    if test_config.encoded_by is not None:
        encoded_data_dir = results_path / f"{test_config.encoded_by}" / "output_data"
        if not encoded_data_dir.exists():
            print("Skipping test because encoded data does not exist")
            continue
    else:
        encoded_data_dir = None

    # Load the models
    model_part_id = (test_config.model_version, test_config.model_type, test_config.target_device)
    if model_part_id not in model_paths:
        print(f"Model not found for {model_part_id}. Skipping test.")
        continue
    model_path = model_paths[model_part_id]
    split_model = load_split_model(model_path, test_config.runtime_params)

    # Run the validation test
    output_data_dir = results_path / f"{test_name}" / "output_data"
    model_tester = ModelTester(split_model=split_model)
    test_results = model_tester.run_validation_test(
        test_config=test_config_path,
        num_clips_limit=test_config.num_clips_limit,
        use_encoder=encoded_data_dir is None,
        use_decoder=test_config.use_decoder,
        output_data_dir=output_data_dir,
        encoded_data_dir=encoded_data_dir,
        iframe_period=test_config.iframe_period,
        metrics_bit_depth=metrics_bit_depth,
        ltr_start_idx=test_config.ltr_start_idx,
        ltr_period=test_config.ltr_period,
        proactive_ltr_recovery=True,
    )
    test_results_path.parent.mkdir(parents=True, exist_ok=True)
    with open(test_results_path, "w") as f:
        json.dump(test_results.to_dict(), f, indent=4)
    test_result_files[test_name] = test_results_path

## Visualize results

In [ ]:
def aggregate_metrics(df):
    return df.groupby(["q_index"]).agg(
        {
            "bpp": "mean",
            "psnr": "mean",
            "psnr_y": "mean",
            "psnr_u": "mean",
            "psnr_v": "mean",
        }
    )


metrics_data = parse_metrics({**comparison_metric_files, **test_result_files})
df_agg_anchor = aggregate_metrics(metrics_data[dataset_config.anchor_name])
df_summary = []

fig, axs = plt.subplots(1, 2, figsize=(18, 6))
plot_idx = 0
for i, (model_name, df_metrics) in enumerate(metrics_data.items()):
    short_model_name = model_name
    df_agg = aggregate_metrics(df_metrics)
    bd_rate = calculate_bd_rate(df_agg_anchor["bpp"], df_agg_anchor["psnr"], df_agg["bpp"], df_agg["psnr"])

    psnr_values = np.array([t.frame_psnr for t in df_metrics.query("q_index == 63").itertuples()])
    mean_psnr_over_frames = psnr_values.mean(axis=0)
    test_config = test_configs.get(model_name)

    # Rate vs distortion
    ax = axs[0]
    color = f"C{plot_idx}"
    ax.plot(
        df_agg["bpp"],
        df_agg["psnr"],
        "o-",
        ms=3,
        color=color,
        label=f"{short_model_name}, BD-rate: {bd_rate:.3f}%",
        alpha=0.7,
    )

    # PSNR over frames
    ax = axs[1]
    if psnr_values.size > 0:
        ax.plot(
            mean_psnr_over_frames,
            color=color,
            label=f"{short_model_name}, PSNR: {np.mean(mean_psnr_over_frames):.3f}",
            alpha=0.7,
        )
    plot_idx += 1

    if test_config is not None:
        df_summary.append(
            {
                "test_name": test_config.test_name,
                "model_version": test_config.model_version,
                "encoder_name": test_config.encoder_name,
                "decoder_name": test_config.decoder_name,
                "encoded_by": test_config.encoded_by,
                "iframe_period": test_config.iframe_period,
                "ltr_start_idx": test_config.ltr_start_idx,
                "ltr_period": test_config.ltr_period,
                "bd_rate": bd_rate,
                "psnr": np.mean(mean_psnr_over_frames),
            }
        )

df_summary = pd.DataFrame(df_summary)

ax = axs[0]
ax.set_xlabel("BPP")
ax.set_ylabel("PSNR (dB)")
ax.set_xscale("log")
ax.set_xlim(0.0005, 0.2)
ax.set_ylim(20, 50)
ax.legend(prop={"size": 9}, loc=4)
ax.grid(True)

ax = axs[1]
ax.legend(loc=4)
ax.set_xlabel("Frame Number")
ax.set_ylabel("PSNR (dB)")
ax.set_ylim(30, 50)
ax.legend(prop={"size": 9}, loc=3)
ax.grid(True)

fig.subplots_adjust(wspace=0.15)

In [ ]:
if test_type == TestType.DIVERGENCE:
    df_pivot = pd.pivot_table(df_summary, index="decoder_name", columns="encoder_name", values="psnr", sort=False)  # type: ignore
    df_pivot = df_pivot.combine_first(df_pivot.T)
    display(df_pivot)
    print(df_pivot.to_csv(sep="\t"))

## Proactive LTR recovery sweep results

In [ ]:
# type: ignore
if test_type == TestType.PROACTIVE_LTR_RECOVERY:
    df_baseline = df_summary[df_summary["encoded_by"].isnull()].set_index("test_name")  # type: ignore
    df_cross_device = df_summary[df_summary["encoded_by"].notnull()]  # type: ignore
    df_divergence_summary = pd.merge(
        left=df_cross_device,
        right=df_baseline[["psnr", "bd_rate"]],
        how="left",
        left_on="encoded_by",
        right_on="test_name",
        suffixes=(None, "_baseline"),
    )

    colors = {32: "C4", 64: "C0", 128: "C1", 192: "C2", 256: "C3"}
    markers = {0: "o", 32: "s", 48: "X", 64: "D", 96: "v", 128: "P"}
    edgecolors = {0: "black", 4: "violet", 8: "blue", 16: "yellow", 24: "cyan"}

    plt.figure(figsize=(12, 8))
    plt.gca().set_axisbelow(True)
    plt.gca().grid(color="gray", linestyle="dashed")

    for t in df_divergence_summary.itertuples():
        plt.plot(
            [t.bd_rate, t.bd_rate_baseline],
            [t.psnr, t.psnr_baseline],
            color=colors[t.iframe_period],
            alpha=0.3,
        )

    plt.scatter(
        df_divergence_summary["bd_rate_baseline"],
        df_divergence_summary["psnr_baseline"],
        marker=".",
        color=[colors[t.iframe_period] for t in df_divergence_summary.itertuples()],
        alpha=0.3,
    )

    for t in df_divergence_summary.itertuples():
        plt.scatter(
            [t.bd_rate],
            [t.psnr],
            marker=markers[t.ltr_period],
            s=80,
            color=colors[t.iframe_period],
            label=f"{t.iframe_period}-{t.ltr_period}-{t.ltr_start_idx}, BD-rate: {t.bd_rate:.1f}%, PSNR: {t.psnr:.1f}",
            edgecolors=edgecolors[t.ltr_start_idx],
            linewidths=1.5,
            alpha=1.0,
        )
    plt.title("DMC 6.1sbr LReLU (CoreML GPU vs CoreML NPU), VCD S1 (40 clips)")
    plt.xlabel("BD-rate (%) vs H265")
    plt.ylabel("Mean PSNR @ QP 63")
    plt.legend(prop={"size": 8.5}, bbox_to_anchor=(1.01, 1.02), loc="upper left")
    plt.xlim(-61, -5)
    plt.ylim(40, 44)

## Qp mapping results

In [ ]:
def map_qp_to_q_index(qp):
    return np.clip(np.round((44.0 - qp) / 0.3), 0, 63).astype(int)


qps = np.arange(52)
q_index_map = map_qp_to_q_index(qps)

# df_h264 = aggregate_metrics(metrics_data["intel_hw_h264_lp"]).reset_index()
df_h265 = aggregate_metrics(metrics_data["intel_hw_hevc_lp"]).reset_index()
df_mlvc = aggregate_metrics(metrics_data["dmc61sbr_lrelu, azureml"])

fig, axs = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axs[0]
# ax.plot(df_h264["q_index"], df_h264["psnr"], "o-", label="H264")
ax.plot(df_h265["q_index"], df_h265["psnr"], "o-", label="H265")
ax.plot(df_mlvc.index, df_mlvc["psnr"], "x-", label="MLVC")
ax.plot(qps, df_mlvc.iloc[q_index_map]["psnr"], "x-", label="MLVC - scaled")

ax.grid(True)
ax.set_xlabel("Qp")
ax.set_ylabel("PSNR (dB)")
ax.legend(loc=1)
# ax.set_xlim(20, 51)
ax.set_ylim(30, 46)

ax = axs[1]
ax.plot(qps, q_index_map, "x-")
ax.set_xlabel("Qp")
ax.set_ylabel("MLVC Q-Index")
ax.grid(True)
ax.set_xlim(0, 51)